# Transformer optimizer benchmark

This notebook runs the optimizer benchmark on a **free Colab T4 GPU**. It clones the support code, runs the tests and experiments, and validates the generated results in the Colab runtime. It contains no download, token, commit, push, or README-update workflow. Expected runtime depends on the assigned Colab host; the verified reference run completed in about six minutes.

## 1. Configuration and checkout

Select **Runtime → Change runtime type → T4 GPU**, then run all cells. The repository is cloned only to obtain the versioned support code.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LokeshJatangi/transformer_optimizer_benchmark.git"
BRANCH = "colab-results"
WORKDIR = Path("/content/transformer-optimizer-benchmark")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
source_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
os.environ["SOURCE_COMMIT"] = source_commit
print("Checked out source commit:", source_commit)

## 2. Environment and unit tests

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch

assert torch.cuda.is_available(), "Enable a GPU runtime"
device_name = torch.cuda.get_device_name(0)
assert "T4" in device_name.upper(), f"Expected a Colab T4, got {device_name}"
print("Validated device:", device_name)
subprocess.run(["python", "-m", "pytest", "-q"], check=True)

## 3. Full synchronized T4 experiment

This is the only cell that creates measured results. Candidate comparisons use identical initialization, batch order, validation data, and tuning budgets.

In [ ]:
from transformer_optimizer_benchmark import run_full

metrics = run_full()
print("Completed in", metrics["timing"]["total_readable"])
print("Retained scheduler:", metrics["scheduler_comparison"]["winner"])
print("Width-4096 LR prediction:", metrics["width_sweep"]["fit"]["predicted_lr_width_4096"])
print("Confidence:", metrics["width_sweep"]["fit"]["confidence"])

## 4. Artifact and assertion audit

In [ ]:
import csv
import json

from transformer_optimizer_benchmark import validate_metrics

results_dir = Path("results")
saved = json.loads((results_dir / "metrics.json").read_text())
validate_metrics(saved)

required = [
    "metrics.json",
    "run_log.json",
    "run_log.csv",
    "adam_bias_correction.png",
    "relative_updates_cosine.png",
    "relative_updates_wsd.png",
    "width_lr_sweep.png",
    "retained_model.pt",
]
missing = [name for name in required if not (results_dir / name).exists()]
assert not missing, missing

run_log = json.loads((results_dir / "run_log.json").read_text())
with (results_dir / "run_log.csv").open(newline="") as handle:
    csv_rows = list(csv.DictReader(handle))
assert run_log == saved["runs"]
assert len(csv_rows) == len(run_log)
assert all(row["status"] == "ok" for row in run_log)
print(f"Validated {len(run_log)} successful timing records and {len(required)} required artifacts.")